## Combination

In [ ]:
import duckdb
# Path setup
p_path = '../data_w/raw/possessions_202526.parquet'
f_path = '../data_w/raw/foul_events_202526.parquet'
s_path = '../data_w/raw/substitution_events_202526.parquet'
t_path = '../data_w/raw/timeout_events_202526.parquet'
query = f"""
WITH events AS (
    -- STEP 1: UNION ALL into a Long/Chronological Format
    -- We assign an 'event_priority' for tie-breaking at the exact same game_clock_secs.
    
    -- Possessions (Has all the rich shot/play data)
    SELECT 
        game_id, 
        period, 
        game_clock_secs,
        1 AS event_priority, 
        'POSSESSION' AS event_type,
        possession_id,
        COALESCE(points, 0) AS points_scored,
        
        -- Possession-specific columns 
        home_lineup_id,
        away_lineup_id,
        play_type,
        shot_distance,
        shot_x,
        home_score,
        
        NULL AS foul_type,
        NULL AS sub_in_id,
        NULL AS timeout_type
    FROM '{p_path}'
    UNION ALL 
    
    -- Fouls
    SELECT 
        game_id, 
        period, 
        game_clock_secs,
        2 AS event_priority,
        'FOUL' AS event_type,
        NULL AS possession_id,
        0 AS points_scored,
        
        -- NULL placeholders for Possession columns
        NULL AS home_lineup_id,
        NULL AS away_lineup_id,
        NULL AS play_type,
        NULL AS shot_distance,
        NULL AS shot_x,
        NULL AS home_score,
        
        foul_type,
        NULL AS sub_in_id,
        NULL AS timeout_type
    FROM '{f_path}'
    UNION ALL 
    
    -- Timeouts
    SELECT 
        game_id, 
        period, 
        game_clock_secs,
        3 AS event_priority,
        'TIMEOUT' AS event_type,
        NULL AS possession_id,
        0 AS points_scored,
        
        -- NULL placeholders
        NULL AS home_lineup_id,
        NULL AS away_lineup_id,
        NULL AS play_type,
        NULL AS shot_distance,
        NULL AS shot_x,
        NULL AS home_score,
        
        NULL AS foul_type,
        NULL AS sub_in_id,
        timeout_type
    FROM '{t_path}'
    UNION ALL 
    
    -- Substitutions
    SELECT 
        game_id, 
        period, 
        game_clock_secs,
        4 AS event_priority,
        'SUBSTITUTION' AS event_type,
        NULL AS possession_id,
        0 AS points_scored,
        
        -- NULL placeholders
        NULL AS home_lineup_id,
        NULL AS away_lineup_id,
        NULL AS play_type,
        NULL AS shot_distance,
        NULL AS shot_x,
        NULL AS home_score,
        
        NULL AS foul_type,
        player_in_id AS sub_in_id,
        NULL AS timeout_type
    FROM '{s_path}'
),
state_tracker AS (
    -- STEP 2: Forward-Fill State and Compute Rolling Metrics
    SELECT 
        *,
        -- Timeout Trackers: Creates a new "group" bucket every time a timeout occurs.
        SUM(CASE WHEN event_type = 'TIMEOUT' THEN 1 ELSE 0 END) OVER current_game_window AS cumulative_timeouts,
        
        -- State Forward-Fill for Substitutions
        LAST_VALUE(sub_in_id IGNORE NULLS) OVER current_game_window AS active_sub_in_id,
        -- State Forward-Fill for Possession-only attributes 
        -- This pushes the context of the previous possession downwards onto foul/timeout ticks!
        LAST_VALUE(possession_id IGNORE NULLS) OVER current_game_window AS current_possession_id,
        LAST_VALUE(home_lineup_id IGNORE NULLS) OVER current_game_window AS current_home_lineup_id,
        LAST_VALUE(away_lineup_id IGNORE NULLS) OVER current_game_window AS current_away_lineup_id,
        LAST_VALUE(home_score IGNORE NULLS) OVER current_game_window AS current_home_score,
        
        -- Look-Back Momentum: Points scored in the last 3 minutes
        SUM(points_scored) OVER (
            PARTITION BY game_id, period
            ORDER BY (720 - game_clock_secs) ASC
            RANGE BETWEEN 180 PRECEDING AND CURRENT ROW
        ) AS points_scored_last_3m
    FROM events
    WINDOW current_game_window AS (
        PARTITION BY game_id
        -- The ORDER BY here natively resolves the simultaneous events problem using our priority integers
        ORDER BY period ASC, game_clock_secs DESC, event_priority ASC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    )
)
-- STEP 3: Output the Continuous Tick-by-Tick Feed!
SELECT 
    game_id, 
    period, 
    game_clock_secs,
    event_type, -- Shows POSSESSION, FOUL, TIMEOUT, etc.
    
    -- Dynamically pull the forward-filled context for ALL rows!
    -- Since the raw column is blank for a foul, we serve up the "current_x" tracked version instead.
    current_possession_id AS possession_id,
    current_home_lineup_id AS home_lineup_id,
    current_away_lineup_id AS away_lineup_id,
    current_home_score AS home_score,
    
    -- Dynamically change play_type for non-possession rows
    -- If play_type is NULL (because it's a foul row), it outputs the event_type ('FOUL') instead.
    COALESCE(play_type, event_type) AS play_type,
    
    -- Play Outcomes (These will still be blank/0 on foul rows, which is correct)
    points_scored,
    shot_distance,
    shot_x,
    
    -- Engineered Variables
    active_sub_in_id,
    cumulative_timeouts,
    points_scored_last_3m
FROM state_tracker
-- WHERE event_type = 'POSSESSION'  (Keep this commented out to see every single event tick!)
ORDER BY game_id, period ASC, game_clock_secs DESC, event_priority ASC;
"""
df_mega = duckdb.query(query).df()
# Handle sparsity for the model
df_mega['active_sub_in_id'] = df_mega['active_sub_in_id'].fillna(0)
# Verify Output
df_mega.head(20)

In [134]:
print(list(df.columns.to_list()))

['game_id', 'possession_id', 'period', 'game_clock_secs', 'period_wall_clock', 'possessing_team', 'team_scored', 'outcome', 'home_score', 'away_score', 'points', 'shot_value', 'shot_x', 'shot_y', 'shot_distance', 'shot_type', 'play_type', 'player_id', 'player_name', 'home_lineup_id', 'away_lineup_id']


In [135]:
import duckdb
import pandas as pd

# Path to your file
file_path = '../data_w/raw/possessions_202526.parquet'

# Load the data using DuckDB
# We'll grab the first 5 rows for a better look
df = duckdb.query(f"SELECT * FROM '{file_path}' LIMIT 5").df()

# View the interactive table
df.head()

,game_id,possession_id,period,game_clock_secs,period_wall_clock,possessing_team,team_scored,outcome,home_score,away_score,...,shot_value,shot_x,shot_y,shot_distance,shot_type,play_type,player_id,player_name,home_lineup_id,away_lineup_id
0,0022400001,1,1,697.0,7:11 PM EST,away,away,stop,0,0,...,0,0,0,0,,Stop,0,,dde33373b842,030440005933
1,0022400001,2,1,682.0,7:11 PM EST,home,home,stop,0,0,...,0,0,0,0,,Stop,0,,dde33373b842,030440005933
2,0022400001,3,1,677.0,7:11 PM EST,away,away,turnover,0,0,...,0,0,0,0,,Turnover,1630552,Johnson,dde33373b842,030440005933
3,0022400001,4,1,657.0,7:11 PM EST,home,home,turnover,0,0,...,0,0,0,0,,Turnover,1627759,Brown,dde33373b842,030440005933
4,0022400001,5,1,655.0,7:11 PM EST,away,away,turnover,0,0,...,0,0,0,0,,Turnover,1630700,Daniels,dde33373b842,030440005933


In [140]:
print(list(df.columns.to_list()))

['game_id', 'possession_id', 'period', 'game_clock_secs', 'period_wall_clock', 'possessing_team', 'team_scored', 'outcome', 'home_score', 'away_score', 'points', 'shot_value', 'shot_x', 'shot_y', 'shot_distance', 'shot_type', 'play_type', 'player_id', 'player_name', 'home_lineup_id', 'away_lineup_id']


In [ ]:
import duckdb
# Path setup
p_path = '../data_w/raw/possessions_202526.parquet'
f_path = '../data_w/raw/foul_events_202526.parquet'
s_path = '../data_w/raw/substitution_events_202526.parquet'
t_path = '../data_w/raw/timeout_events_202526.parquet'
query = f"""
WITH events AS (
    -- STEP 1: UNION ALL into a Long/Chronological Format
    -- We assign an 'event_priority' for tie-breaking at the exact same game_clock_secs.
    
    -- Possessions (Has all the rich shot/play data)
    SELECT 
        -- ADDED ONES 
        home_score,
        away_score,
        possessing_team,

        -- END
        game_id, 
        period, 
        game_clock_secs,
        1 AS event_priority, 
        'POSSESSION' AS event_type,
        possession_id,
        COALESCE(points, 0) AS points_scored,
        
        -- Possession-specific columns 
        home_lineup_id,
        away_lineup_id,
        play_type,
        shot_distance,
        shot_x,
        home_score,
        
        NULL AS foul_type,
        NULL AS sub_in_id,
        NULL AS timeout_type
    FROM '{p_path}'
    UNION ALL 
    
    -- Fouls
    SELECT 
        game_id, 
        period, 
        game_clock_secs,
        2 AS event_priority,
        'FOUL' AS event_type,
        NULL AS possession_id,
        0 AS points_scored,
        
        -- NULL placeholders for Possession columns
        NULL AS home_lineup_id,
        NULL AS away_lineup_id,
        NULL AS play_type,
        NULL AS shot_distance,
        NULL AS shot_x,
        NULL AS home_score,
        
        foul_type,
        NULL AS sub_in_id,
        NULL AS timeout_type
    FROM '{f_path}'
    UNION ALL 
    
    -- Timeouts
    SELECT 
        game_id, 
        period, 
        game_clock_secs,
        3 AS event_priority,
        'TIMEOUT' AS event_type,
        NULL AS possession_id,
        0 AS points_scored,
        
        -- NULL placeholders
        NULL AS home_lineup_id,
        NULL AS away_lineup_id,
        NULL AS play_type,
        NULL AS shot_distance,
        NULL AS shot_x,
        NULL AS home_score,
        
        NULL AS foul_type,
        NULL AS sub_in_id,
        timeout_type
    FROM '{t_path}'
    UNION ALL 
    
    -- Substitutions
    SELECT 
        game_id, 
        period, 
        game_clock_secs,
        4 AS event_priority,
        'SUBSTITUTION' AS event_type,
        NULL AS possession_id,
        0 AS points_scored,
        
        -- NULL placeholders
        NULL AS home_lineup_id,
        NULL AS away_lineup_id,
        NULL AS play_type,
        NULL AS shot_distance,
        NULL AS shot_x,
        NULL AS home_score,
        
        NULL AS foul_type,
        player_in_id AS sub_in_id,
        NULL AS timeout_type
    FROM '{s_path}'
),
state_tracker AS (
    -- STEP 2: Forward-Fill State and Compute Rolling Metrics
    SELECT 
        *,
        -- Timeout Trackers: Creates a new "group" bucket every time a timeout occurs.
        SUM(CASE WHEN event_type = 'TIMEOUT' THEN 1 ELSE 0 END) OVER current_game_window AS cumulative_timeouts,
        
        -- State Forward-Fill for Substitutions
        LAST_VALUE(sub_in_id IGNORE NULLS) OVER current_game_window AS active_sub_in_id,
        -- State Forward-Fill for Possession-only attributes 
        -- This pushes the context of the previous possession downwards onto foul/timeout ticks!
        LAST_VALUE(possession_id IGNORE NULLS) OVER current_game_window AS current_possession_id,
        LAST_VALUE(home_lineup_id IGNORE NULLS) OVER current_game_window AS current_home_lineup_id,
        LAST_VALUE(away_lineup_id IGNORE NULLS) OVER current_game_window AS current_away_lineup_id,
        LAST_VALUE(home_score IGNORE NULLS) OVER current_game_window AS current_home_score,
        
        -- Look-Back Momentum: Points scored in the last 3 minutes
        SUM(points_scored) OVER (
            PARTITION BY game_id, period
            ORDER BY (720 - game_clock_secs) ASC
            RANGE BETWEEN 180 PRECEDING AND CURRENT ROW
        ) AS points_scored_last_3m
    FROM events
    WINDOW current_game_window AS (
        PARTITION BY game_id
        -- The ORDER BY here natively resolves the simultaneous events problem using our priority integers
        ORDER BY period ASC, game_clock_secs DESC, event_priority ASC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    )
)
-- STEP 3: Output the Continuous Tick-by-Tick Feed!
SELECT 
    game_id, 
    period, 
    game_clock_secs,
    event_type, -- Shows POSSESSION, FOUL, TIMEOUT, etc.
    
    -- Dynamically pull the forward-filled context for ALL rows!
    -- Since the raw column is blank for a foul, we serve up the "current_x" tracked version instead.
    current_possession_id AS possession_id,
    current_home_lineup_id AS home_lineup_id,
    current_away_lineup_id AS away_lineup_id,
    current_home_score AS home_score,
    
    -- Dynamically change play_type for non-possession rows
    -- If play_type is NULL (because it's a foul row), it outputs the event_type ('FOUL') instead.
    COALESCE(play_type, event_type) AS play_type,
    
    -- Play Outcomes (These will still be blank/0 on foul rows, which is correct)
    points_scored,
    shot_distance,
    shot_x,
    
    -- Engineered Variables
    active_sub_in_id,
    cumulative_timeouts,
    points_scored_last_3m
FROM state_tracker
-- WHERE event_type = 'POSSESSION'  (Keep this commented out to see every single event tick!)
ORDER BY game_id, period ASC, game_clock_secs DESC, event_priority ASC;
"""
df_mega = duckdb.query(query).df()
# Handle sparsity for the model
df_mega['active_sub_in_id'] = df_mega['active_sub_in_id'].fillna(0)
# Verify Output
df_mega.head(20)

,game_id,period,game_clock_secs,event_type,possession_id,home_lineup_id,away_lineup_id,home_score,play_type,points_scored,shot_distance,shot_x,active_sub_in_id,cumulative_timeouts,points_scored_last_3m
0,0022400001,1,697.0,POSSESSION,1,dde33373b842,030440005933,0,Stop,0,0,0,0,0.0,0.0
1,0022400001,1,682.0,POSSESSION,2,dde33373b842,030440005933,0,Stop,0,0,0,0,0.0,0.0
2,0022400001,1,677.0,POSSESSION,3,dde33373b842,030440005933,0,Turnover,0,0,0,0,0.0,0.0
3,0022400001,1,657.0,POSSESSION,4,dde33373b842,030440005933,0,Turnover,0,0,0,0,0.0,0.0
4,0022400001,1,655.0,POSSESSION,5,dde33373b842,030440005933,0,Turnover,0,0,0,0,0.0,0.0
5,0022400001,1,654.0,POSSESSION,6,dde33373b842,030440005933,0,Turnover,0,0,0,0,0.0,0.0
6,0022400001,1,650.0,POSSESSION,7,dde33373b842,030440005933,0,Made Shot,3,26,157,0,0.0,3.0
7,0022400001,1,635.0,POSSESSION,8,dde33373b842,030440005933,3,Made Shot,3,27,102,0,0.0,6.0
8,0022400001,1,624.0,POSSESSION,9,dde33373b842,030440005933,3,Free Throw,0,0,0,0,0.0,6.0
9,0022400001,1,624.0,FOUL,9,dde33373b842,030440005933,3,FOUL,0,<NA>,<NA>,0,0.0,6.0
